Make sure top left says `traffic` and not `Select Kernel`

In [ ]:
!uv pip install kaggle

In [1]:
import os
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import zipfile

In [2]:
subprocess.run(["kaggle", "datasets", "download", "-d", "liuxu77/largest", "-p", "../data/external/LargeST/data/ca/", "--unzip"])

CompletedProcess(args=['kaggle', 'datasets', 'download', '-d', 'liuxu77/largest', '-p', '../data/external/LargeST/data/ca/', '--unzip'], returncode=0)

In [3]:
# Remove the zip file if it exists
zip_path = Path("../data/external/LargeST/data/ca/largest.zip")
if zip_path.exists():
    os.remove(zip_path)

# Remove the following files if they exist
files_to_remove = [
    Path("../data/external/LargeST/data/ca/ca_his_raw_2020.h5"),
    Path("../data/external/LargeST/data/ca/ca_his_raw_2021.h5"),
]
for file_path in files_to_remove:
    if file_path.exists():
        os.remove(file_path)

In [4]:
os.makedirs('../data/external/LargeST/data/sd/largest/', exist_ok=True)

In [5]:
ca_meta = pd.read_csv('../data/external/LargeST/data/ca/ca_meta.csv')
sd_meta = ca_meta[ca_meta.District == 11]
sd_meta = sd_meta.reset_index()
sd_meta = sd_meta.drop(columns=['index'])
sd_meta.to_csv('../data/external/LargeST/data/sd/largest/sd_meta.csv', index=False)
print(sd_meta[sd_meta.duplicated(subset=['Lat', 'Lng'])])
sd_meta

Empty DataFrame
Columns: [ID, Lat, Lng, District, County, Fwy, Lanes, Type, Direction, ID2]
Index: []


,ID,Lat,Lng,District,County,Fwy,Lanes,Type,Direction,ID2
0,1114091,32.544463,-117.032486,11,San Diego,I5-N,6,Mainline,N,6931
1,1118333,32.551690,-117.045725,11,San Diego,I5-N,4,Mainline,N,6932
2,1118348,32.558459,-117.061845,11,San Diego,I5-N,4,Mainline,N,6933
3,1114720,32.561334,-117.067081,11,San Diego,I5-N,4,Mainline,N,6934
4,1118352,32.569052,-117.076507,11,San Diego,I5-N,4,Mainline,N,6935
...,...,...,...,...,...,...,...,...,...,...
711,1123276,32.564373,-116.969279,11,San Diego,I905-W,3,Mainline,W,7642
712,1122394,32.564336,-116.961909,11,San Diego,I905-W,3,Mainline,W,7643
713,1123261,32.564112,-116.950744,11,San Diego,I905-W,3,Mainline,W,7644
714,1123256,32.560935,-116.944559,11,San Diego,I905-W,3,Mainline,W,7645


In [6]:
sd_meta_id2 = sd_meta.ID2.values.tolist()
print(len(sd_meta_id2))

ca_rn_adj = np.load('../data/external/LargeST/data/ca/ca_rn_adj.npy')
print(ca_rn_adj.shape)

sd_rn_adj = ca_rn_adj[sd_meta_id2]
sd_rn_adj = sd_rn_adj[:,sd_meta_id2]
print(sd_rn_adj.shape)

np.save('../data/external/LargeST/data/ca/sd_rn_adj.npy', sd_rn_adj)

716
(8600, 8600)
(716, 716)


In [7]:
years = ['2017', '2018', '2019']

sd_meta.ID = sd_meta.ID.astype(str)
sd_meta_id = sd_meta.ID.values.tolist()

for year in years:
    ca_his = pd.read_hdf('../data/external/LargeST/data/ca/ca_his_raw_' + year +'.h5')
    sd_his = ca_his[sd_meta_id]
    sd_his.to_hdf('../data/external/LargeST/data/sd/largest/sd_his_' + year + '.h5', key='t', mode='w')

In [8]:
os.makedirs('../data/pollution/no2', exist_ok=True)
os.makedirs('../data/pollution/co', exist_ok=True)
os.makedirs('../data/pollution/pm25', exist_ok=True)

In [10]:
env_sources = {
    "no2" : [
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42602_2017.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42602_2018.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42602_2019.zip",
    ],
    "co" : [
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42101_2017.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42101_2018.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42101_2019.zip",
    ],
    "pm25" : [
        "https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2017.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2018.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2019.zip",
    ]
}

env_path_files = {
    "no2" : [],
    "co" : [],
    "pm25" : []
}

for env, sources in env_sources.items():
    for source in sources:
        filename = source.split("/")[-1]
        save_path = f"../data/pollution/{env}/{filename}"
        env_path_files[env].append(save_path)
#        subprocess.run(["curl", "-L", "-o", save_path, source])

        # Unzip the file
#        with zipfile.ZipFile(save_path, 'r') as zip_ref:
#            zip_ref.extractall(f"../data/pollution/{env}/")
        # Remove the zip file
#        if os.path.exists(save_path):
#            os.remove(save_path)
        

In [13]:
# Iterate through the env_path_files dictionary and read each CSV file into a DataFrame
env_dataframes = {}
for env, paths in env_path_files.items():
    dfs = []
    for path in paths:
        # Replace the .zip extension with .csv to get the actual CSV file path
        csv_path = path.replace('.zip', '.csv')
        df = pd.read_csv(csv_path)
        dfs.append(df)
    env_dataframes[env] = pd.concat(dfs, ignore_index=True)

C:\Users\iason\AppData\Local\Temp\ipykernel_30276\1273014564.py:8: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


In [ ]:
for env_kpi in env_dataframes.keys():
    curr_df = env_dataframes[env_kpi]
    # Keep only rows where state code is 6 (California) and county code is 73 (San Diego County)
    curr_df = curr_df[(curr_df['State Code'] == 6) & (curr_df['County Code'] == 73)]
    # Drop nan columns
    curr_df = curr_df.dropna(axis=1, how='all')
    # Validate that data exists
    if curr_df.empty:
        print(f"No data available for {env_kpi} in San Diego County.")
    else:
        # Save the filtered DataFrame back to the directory
        curr_df.to_csv(f"../data/pollution/{env_kpi}/sd_{env_kpi}.csv", index=False)